In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col,sum, when, month, expr,year,month, to_date, date_format, avg, count, sum
from pyspark.sql.types import *

In [ ]:
spark = SparkSession.builder.getOrCreate()

In [ ]:
clientes = spark.read.csv('drive/MyDrive/trabalho_ebac_pyspark/01_dados_brutos/13a_clientes_p.csv', header=True).limit(5)
produtos = spark.read.csv('drive/MyDrive/trabalho_ebac_pyspark/01_dados_brutos/14a_produtos_p.csv', header=True).limit(5)
avaliacoes = spark.read.csv('drive/MyDrive/trabalho_ebac_pyspark/01_dados_brutos/11a_avaliacoes.csv',header=True).limit(5)
sessoes = spark.read.csv('drive/MyDrive/trabalho_ebac_pyspark/01_dados_brutos/12a_sessoes.csv', header=True).limit(5)
transacoes = spark.read.csv('drive/MyDrive/trabalho_ebac_pyspark/01_dados_brutos/15a_transações.csv', header=True).limit(5)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
clientes.printSchema()
clientes.show()
produtos.printSchema()
produtos.show()
avaliacoes.printSchema()
avaliacoes.show(truncate=False)
sessoes.printSchema()
sessoes.show()
transacoes.printSchema()
transacoes.show()

root
 |-- id_cliente: string (nullable = true)
 |-- data_cadastro: string (nullable = true)
 |-- idade: string (nullable = true)
 |-- genero: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- cancelou: string (nullable = true)
 |-- valor_vitalicio: string (nullable = true)
 |-- aceita_email: string (nullable = true)
 |-- possui_app: string (nullable = true)

+----------+-------------+-----+-----------------+--------------+-------------------+--------+---------------+------------+----------+
|id_cliente|data_cadastro|idade|           genero|          pais|           segmento|cancelou|valor_vitalicio|aceita_email|possui_app|
+----------+-------------+-----+-----------------+--------------+-------------------+--------+---------------+------------+----------+
|    C00000|   2020-02-13|   28|        masculino|        Brasil|            Premium|       0|        1595.27|           0|         0|
|    C00001|   2021-10-01|   22|prefiro nao

In [ ]:
# 1. clientes
schema_clientes = StructType([
    StructField("id_cliente", StringType(), True),
    StructField("data_cadastro", DateType(), True),
    StructField("idade", ShortType(), True),
    StructField("genero", StringType(), True),
    StructField("pais", StringType(), True),
    StructField("segmento", StringType(), True),
    StructField("cancelou", ShortType(), True),
    StructField("valor_vitalicio", DoubleType(), True),
    StructField("aceita_email", ShortType(), True),
    StructField("possui_app", ShortType(), True)
])


# 2. PRODUTOS
schema_produtos = StructType([
    StructField("id_produto", StringType(), True),
    StructField("nome_produto", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("marca", StringType(), True),
    StructField("preco", DoubleType(), True),
    StructField("classificacao_media", FloatType(), True),
    StructField("qtd_avaliacoes", ShortType(), True),
    StructField("qtd_estoque", ShortType(), True),
    StructField("desconto_pct", ShortType(), True),
    StructField("destaque", ShortType(), True)
])


# 3. AVALIAÇÕES
schema_avaliacoes = StructType([
    StructField("id_avaliacao", StringType(), True),
    StructField("id_cliente", StringType(), True),
    StructField("id_produto", StringType(), True),
    StructField("data_avaliacao", DateType(), True),
    StructField("classificacao", ShortType(), True),
    StructField("texto_avaliacao", StringType(), True),
    StructField("votos_uteis", ShortType(), True),
    StructField("compra_verificada", ShortType(), True)
])



# 4. SESSÕES
schema_sessoes = StructType([
    StructField("id_sessao", StringType(), True),
    StructField("id_cliente", StringType(), True),
    StructField("data_sessao", TimestampType(), True),
    StructField("dispositivo", StringType(), True),
    StructField("canal", StringType(), True),
    StructField("duracao_segundos", DoubleType(), True),
    StructField("paginas_visualizadas", LongType(), True),
    StructField("converteu", LongType(), True),
    StructField("rejeitou", LongType(), True),
    StructField("adicoes_carrinho", LongType(), True)
])

# 5. TRANSAÇÕES
schema_transacoes = StructType([
    StructField("id_transacao", StringType(), True),
    StructField("id_cliente", StringType(), True),
    StructField("id_produto", StringType(), True),
    StructField("data_transacao", TimestampType(), True),
    StructField("quantidade", ShortType(), True),
    StructField("preco_unitario", DoubleType(), True),
    StructField("valor_total", DoubleType(), True),
    StructField("desconto_aplicado", ShortType(), True),
    StructField("status", StringType(), True),
    StructField("metodo_pagamento", StringType(), True),
    StructField("custo_envio", DoubleType(), True)
])


In [ ]:
clientes = spark.read.csv('drive/MyDrive/trabalho_ebac_pyspark/01_dados_brutos/13a_clientes_p.csv', header=True, schema=schema_clientes)
produtos = spark.read.csv('drive/MyDrive/trabalho_ebac_pyspark/01_dados_brutos/14a_produtos_p.csv', header=True,schema=schema_produtos)
avaliacoes = spark.read.csv('drive/MyDrive/trabalho_ebac_pyspark/01_dados_brutos/11a_avaliacoes.csv',header=True, schema=schema_avaliacoes)
sessoes = spark.read.csv('drive/MyDrive/trabalho_ebac_pyspark/01_dados_brutos/12a_sessoes.csv', header=True, schema=schema_sessoes)
transacoes = spark.read.csv('drive/MyDrive/trabalho_ebac_pyspark/01_dados_brutos/15a_transações.csv', header=True, schema=schema_transacoes)



In [ ]:
clientes.printSchema()
clientes.show(3)
produtos.printSchema()
produtos.show(3)
avaliacoes.printSchema()
avaliacoes.show(n=3,truncate=False)
sessoes.printSchema()
sessoes.show(3)
transacoes.printSchema()
transacoes.show(3)

root
 |-- id_cliente: string (nullable = true)
 |-- data_cadastro: date (nullable = true)
 |-- idade: short (nullable = true)
 |-- genero: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- cancelou: short (nullable = true)
 |-- valor_vitalicio: double (nullable = true)
 |-- aceita_email: short (nullable = true)
 |-- possui_app: short (nullable = true)

+----------+-------------+-----+-----------------+--------------+--------+--------+---------------+------------+----------+
|id_cliente|data_cadastro|idade|           genero|          pais|segmento|cancelou|valor_vitalicio|aceita_email|possui_app|
+----------+-------------+-----+-----------------+--------------+--------+--------+---------------+------------+----------+
|    C00000|   2020-02-13|   28|        masculino|        Brasil| Premium|       0|        1595.27|           0|         0|
|    C00001|   2021-10-01|   22|prefiro nao dizer|        Franca| Regular|       0|        11

In [ ]:
clientes.show()

+----------+-------------+-----+-----------------+--------------+-------------------+--------+---------------+------------+----------+
|id_cliente|data_cadastro|idade|           genero|          pais|           segmento|cancelou|valor_vitalicio|aceita_email|possui_app|
+----------+-------------+-----+-----------------+--------------+-------------------+--------+---------------+------------+----------+
|    C00000|   2020-02-13|   28|        masculino|        Brasil|            Premium|       0|        1595.27|           0|         0|
|    C00001|   2021-10-01|   22|prefiro nao dizer|        Franca|            Regular|       0|        1160.61|           1|         0|
|    C00002|   2022-06-26|   30|         feminino|Estados_Unidos|                VIP|       0|        3093.32|           1|         1|
|    C00003|   2021-12-21|   48|         feminino|Estados_Unidos|            Premium|       1|        2131.08|           1|         0|
|    C00004|   2021-09-16|   37|        masculino|     

In [ ]:
clientes.show(3)
produtos.show(3)
avaliacoes.show(n=3,truncate=False)
sessoes.show(3)
transacoes.show(3)

+----------+-------------+-----+-----------------+--------------+--------+--------+---------------+------------+----------+
|id_cliente|data_cadastro|idade|           genero|          pais|segmento|cancelou|valor_vitalicio|aceita_email|possui_app|
+----------+-------------+-----+-----------------+--------------+--------+--------+---------------+------------+----------+
|    C00000|   2020-02-13|   28|        masculino|        Brasil| Premium|       0|        1595.27|           0|         0|
|    C00001|   2021-10-01|   22|prefiro nao dizer|        Franca| Regular|       0|        1160.61|           1|         0|
|    C00002|   2022-06-26|   30|         feminino|Estados_Unidos|     VIP|       0|        3093.32|           1|         1|
+----------+-------------+-----+-----------------+--------------+--------+--------+---------------+------------+----------+
only showing top 3 rows
+----------+--------------------+-------------------+---------+-----+-------------------+--------------+----

In [ ]:
#0. PREPARAÇÃO DA TABELA DE SESSÕES (Criação de data_fim_sessao)
# ==============================================================================
# Soma a duracao_segundos à data_sessao para criar a janela exata de tempo da navegação
sessoes_com_fim = sessoes.withColumn(
    "data_fim_sessao",
    F.from_unixtime(F.unix_timestamp("data_sessao") + F.col("duracao_segundos")).cast("timestamp")
)

# ==============================================================================
# 1. JOIN POR JANELA TEMPORAL (Sessão Exata do Momento da Compra)
# ==============================================================================
# Garante a busca do canal e dispositivo sem multiplicar transações
transacoes_com_sessao = transacoes.alias("t").join(
    sessoes_com_fim.alias("s"),
    on=(
        (F.col("t.id_cliente") == F.col("s.id_cliente")) &
        (F.col("t.data_transacao") >= F.col("s.data_sessao")) &
        (F.col("t.data_transacao") <= F.col("s.data_fim_sessao"))
    ),
    how="left"
).select(
    # Dados da Transação
    F.col("t.id_transacao"),
    F.col("t.id_cliente"),
    F.col("t.id_produto"),
    F.col("t.data_transacao"),
    F.col("t.quantidade"),
    F.col("t.preco_unitario"),
    F.col("t.valor_total"),
    F.col("t.desconto_aplicado"),
    F.col("t.status"),
    F.col("t.metodo_pagamento"),
    F.col("t.custo_envio"),

    # Dados da sessão pontual (capturados no instante exato da compra)
    F.col("s.dispositivo"),
    F.col("s.canal")
)

# ==============================================================================
# 2. AGREGAR SESSÕES POR CLIENTE (Histórico Geral de Comportamento)
# ==============================================================================
# Garante exatamente 1 linha por id_cliente para trazer o perfil sem duplicar o JOIN
sessoes_agregadas = sessoes.groupBy("id_cliente").agg(
    F.count("id_sessao").alias("total_sessoes"),
    F.avg("duracao_segundos").alias("media_duracao_segundos"),
    F.sum("paginas_visualizadas").alias("total_paginas_visualizadas"),
    F.sum("adicoes_carrinho").alias("total_adicoes_carrinho"),
    F.sum("converteu").alias("total_conversoes"),
    F.sum("rejeitou").alias("total_rejeicoes")
)

# ==============================================================================
# 3. JOIN EM CADEIA (Consolidação da OBT - One Big Table)
# ==============================================================================
df_consolidado = transacoes_com_sessao.alias("ts") \
    .join(clientes.alias("c"), "id_cliente", "inner") \
    .join(produtos.alias("p"), "id_produto", "inner") \
    .join(sessoes_agregadas.alias("sa"), "id_cliente", "left") \
    .join(avaliacoes.alias("a"), ["id_cliente", "id_produto"], "left") \
    .select(
        # Identificadores e Transação
        F.col("ts.id_transacao"),
        F.col("id_cliente"),
        F.col("id_produto"),
        F.col("ts.data_transacao"),
        F.col("ts.quantidade"),
        F.col("ts.preco_unitario"),
        F.col("ts.valor_total"),
        F.col("ts.desconto_aplicado"),
        F.col("ts.status"),
        F.col("ts.metodo_pagamento"),
        F.col("ts.custo_envio"),

        # Perfil do Cliente
        F.col("c.data_cadastro"),
        F.col("c.idade"),
        F.col("c.genero"),
        F.col("c.pais"),
        F.col("c.segmento"),
        F.col("c.cancelou"),
        F.col("c.valor_vitalicio"),
        F.col("c.possui_app"),

        # Produto
        F.col("p.categoria"),
        F.col("p.marca"),
        F.col("p.preco").alias("preco_catalogo_produto"),

        # Sessão Pontual da Compra (Dispositivo e Canal no instante da compra)
        F.col("ts.dispositivo"),
        F.col("ts.canal"),

        # Histórico Agregado do Cliente (Comportamento acumulado)
        F.col("sa.total_sessoes"),
        F.col("sa.media_duracao_segundos"),
        F.col("sa.total_paginas_visualizadas"),
        F.col("sa.total_adicoes_carrinho"),
        F.col("sa.total_conversoes"),
        F.col("sa.total_rejeicoes"),

        # Avaliação do Produto pelo Cliente
        F.col("a.classificacao").alias("avaliacao_nota")
    )

In [ ]:
df_consolidado.show(3)



+------------+----------+----------+-------------------+----------+--------------+-----------+-----------------+-----------+----------------+-----------+-------------+-----+---------+--------------+-------------------+--------+---------------+----------+-----------+----------+----------------------+-----------+-----+-------------+----------------------+--------------------------+----------------------+----------------+---------------+--------------+
|id_transacao|id_cliente|id_produto|     data_transacao|quantidade|preco_unitario|valor_total|desconto_aplicado|     status|metodo_pagamento|custo_envio|data_cadastro|idade|   genero|          pais|           segmento|cancelou|valor_vitalicio|possui_app|  categoria|     marca|preco_catalogo_produto|dispositivo|canal|total_sessoes|media_duracao_segundos|total_paginas_visualizadas|total_adicoes_carrinho|total_conversoes|total_rejeicoes|avaliacao_nota|
+------------+----------+----------+-------------------+----------+--------------+----------

In [ ]:
print(f"📊 qtd de linhas: {df_consolidado.count()} | qtd de colunas: {len(df_consolidado.columns)}")

📊 qtd de linhas: 120000 | qtd de colunas: 31


In [ ]:
df_consolidado.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_consolidado.columns]).show()


+------------+----------+----------+--------------+----------+--------------+-----------+-----------------+------+----------------+-----------+-------------+-----+------+----+--------+--------+---------------+----------+---------+-----+----------------------+-----------+------+-------------+----------------------+--------------------------+----------------------+----------------+---------------+--------------+
|id_transacao|id_cliente|id_produto|data_transacao|quantidade|preco_unitario|valor_total|desconto_aplicado|status|metodo_pagamento|custo_envio|data_cadastro|idade|genero|pais|segmento|cancelou|valor_vitalicio|possui_app|categoria|marca|preco_catalogo_produto|dispositivo| canal|total_sessoes|media_duracao_segundos|total_paginas_visualizadas|total_adicoes_carrinho|total_conversoes|total_rejeicoes|avaliacao_nota|
+------------+----------+----------+--------------+----------+--------------+-----------+-----------------+------+----------------+-----------+-------------+-----+------+--

In [ ]:
# ==============================================================================
# TRATAMENTO DE NULOS - VERSÃO PROFISSIONAL
# ==============================================================================

# 1. Mapeamento de métricas numéricas de navegação (NULL -> 0)
colunas_metricas_zero = [
    "total_sessoes",
    "media_duracao_segundos",
    "total_paginas_visualizadas",
    "total_adicoes_carrinho",
    "total_conversoes",
    "total_rejeicoes"
]

# 2. Mapeamento de atributos categóricos da sessão
atributos_sessao = {
    "dispositivo": "nao_identificado",
    "canal": "nao_identificado"
}

# Aplicando os preenchimentos no DataFrame
df_tratado = df_consolidado \
    .fillna({c: 0 for c in colunas_metricas_zero}) \
    .fillna(atributos_sessao)

# 3. Tratamento da avaliação (cria rótulo sem alterar o tipo numérico da nota)
df_tratado = df_tratado.withColumn(
    "avaliacao_nota_label",
    F.when(F.col("avaliacao_nota").isNull(), "nao_avaliado")
     .otherwise(F.col("avaliacao_nota").cast("string"))
)

# 4. Validação final dos nulos restantes
print("📊 Nulos restantes após tratamento:")
df_tratado.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_tratado.columns
]).show(vertical=True)

print("✅ Tratamento de nulos concluído com sucesso!")

📊 Nulos restantes após tratamento:
-RECORD 0----------------------------
 id_transacao               | 0      
 id_cliente                 | 0      
 id_produto                 | 0      
 data_transacao             | 0      
 quantidade                 | 0      
 preco_unitario             | 0      
 valor_total                | 0      
 desconto_aplicado          | 0      
 status                     | 0      
 metodo_pagamento           | 0      
 custo_envio                | 0      
 data_cadastro              | 0      
 idade                      | 0      
 genero                     | 0      
 pais                       | 0      
 segmento                   | 0      
 cancelou                   | 0      
 valor_vitalicio            | 0      
 possui_app                 | 0      
 categoria                  | 0      
 marca                      | 0      
 preco_catalogo_produto     | 0      
 dispositivo                | 0      
 canal                      | 0      
 total_sessoes 

In [ ]:
df_tratado.show(3)

+------------+----------+----------+-------------------+----------+--------------+-----------+-----------------+-----------+----------------+-----------+-------------+-----+---------+--------------+-------------------+--------+---------------+----------+-----------+----------+----------------------+----------------+----------------+-------------+----------------------+--------------------------+----------------------+----------------+---------------+--------------+--------------------+
|id_transacao|id_cliente|id_produto|     data_transacao|quantidade|preco_unitario|valor_total|desconto_aplicado|     status|metodo_pagamento|custo_envio|data_cadastro|idade|   genero|          pais|           segmento|cancelou|valor_vitalicio|possui_app|  categoria|     marca|preco_catalogo_produto|     dispositivo|           canal|total_sessoes|media_duracao_segundos|total_paginas_visualizadas|total_adicoes_carrinho|total_conversoes|total_rejeicoes|avaliacao_nota|avaliacao_nota_label|
+------------+----

In [ ]:
# Mostrar todas as linhas que são duplicadas (completas)
linhas_duplicadas = df_tratado.exceptAll(df_tratado.dropDuplicates())
linhas_duplicadas.show()

# Contar quantas linhas duplicadas existem
print(f"Qtd Linhas duplicadas: {linhas_duplicadas.count()}")

+------------+----------+----------+--------------+----------+--------------+-----------+-----------------+------+----------------+-----------+-------------+-----+------+----+--------+--------+---------------+----------+---------+-----+----------------------+-----------+-----+-------------+----------------------+--------------------------+----------------------+----------------+---------------+--------------+--------------------+
|id_transacao|id_cliente|id_produto|data_transacao|quantidade|preco_unitario|valor_total|desconto_aplicado|status|metodo_pagamento|custo_envio|data_cadastro|idade|genero|pais|segmento|cancelou|valor_vitalicio|possui_app|categoria|marca|preco_catalogo_produto|dispositivo|canal|total_sessoes|media_duracao_segundos|total_paginas_visualizadas|total_adicoes_carrinho|total_conversoes|total_rejeicoes|avaliacao_nota|avaliacao_nota_label|
+------------+----------+----------+--------------+----------+--------------+-----------+-----------------+------+----------------+-

In [ ]:
# ==============================================================================
# ANÁLISE DE PREÇOS - CATÁLOGO vs VENDAS
# ==============================================================================

print("📊 Estatísticas do Catálogo de Produtos (preço cadastrado):")
df_tratado.select("preco_catalogo_produto").distinct().select(
    F.avg("preco_catalogo_produto").alias("media_catalogo"),
    F.min("preco_catalogo_produto").alias("min_catalogo"),
    F.percentile_approx("preco_catalogo_produto", 0.25).alias("p25_catalogo"),
    F.percentile_approx("preco_catalogo_produto", 0.50).alias("mediana_catalogo"),
    F.percentile_approx("preco_catalogo_produto", 0.75).alias("p75_catalogo"),
    F.max("preco_catalogo_produto").alias("max_catalogo")
).show()

print("\n📊 Estatísticas do Preço Praticado nas Vendas (com descontos):")
df_tratado.select(
    F.avg("preco_unitario").alias("media_venda"),
    F.min("preco_unitario").alias("min_venda"),
    F.percentile_approx("preco_unitario", 0.25).alias("p25_venda"),
    F.percentile_approx("preco_unitario", 0.50).alias("mediana_venda"),
    F.percentile_approx("preco_unitario", 0.75).alias("p75_venda"),
    F.max("preco_unitario").alias("max_venda")
).show()

📊 Estatísticas do Catálogo de Produtos (preço cadastrado):
+----------------+------------+------------+----------------+------------+------------+
|  media_catalogo|min_catalogo|p25_catalogo|mediana_catalogo|p75_catalogo|max_catalogo|
+----------------+------------+------------+----------------+------------+------------+
|59.2263924050633|        3.71|       20.23|           37.18|        70.8|      520.45|
+----------------+------------+------------+----------------+------------+------------+


📊 Estatísticas do Preço Praticado nas Vendas (com descontos):
+------------------+---------+---------+-------------+---------+---------+
|       media_venda|min_venda|p25_venda|mediana_venda|p75_venda|max_venda|
+------------------+---------+---------+-------------+---------+---------+
|53.502234666667896|     2.68|    18.52|        33.23|    64.44|   520.45|
+------------------+---------+---------+-------------+---------+---------+



In [ ]:
# =============================================================================
# CRIAÇÃO DE COLUNAS DERIVADAS (MANIPULAÇÃO DE DADOS)
# ==============================================================================
#
# 1. mes_transacao:
#    - Origem: data_transacao
#    - Objetivo: Análise de sazonalidade mensal (ex: picos em dezembro)
#
# 2. faixa_preco_produto:
#    - Origem: preco_catalogo_produto (preço efetivamente vendido)
#    - Critérios (baseados nos percentis de vendas reais):
#        * Baixo Custo: ≤ R$ 33,00 (≈50% das vendas)
#        * Médio Custo: R$ 33,01 a R$ 65,00
#        * Alto Custo: > R$ 65,00
#    - Justificativa: Segmentação baseada em vendas reais, não no catálogo
# ==============================================================================

# ==============================================================================

df_tratado = df_tratado.withColumn('mes_transacao', month(col('data_transacao'))) \
                       .withColumn('faixa_preco_produto',
                                   when(col('preco_catalogo_produto') <= 33.0, 'Baixo Custo')
                                   .when((col('preco_catalogo_produto') > 33.0) & (col('preco_catalogo_produto') <= 65.0), 'Médio Custo')
                                   .otherwise('Alto Custo'))

# Conferindo a distribuição
df_tratado.groupBy('faixa_preco_produto').count().show()

+-------------------+-----+
|faixa_preco_produto|count|
+-------------------+-----+
|        Baixo Custo|55226|
|        Médio Custo|32482|
|         Alto Custo|32292|
+-------------------+-----+



In [ ]:
df_tratado.show(3)

+------------+----------+----------+-------------------+----------+--------------+-----------+-----------------+-----------+----------------+-----------+-------------+-----+---------+--------------+-------------------+--------+---------------+----------+-----------+----------+----------------------+----------------+----------------+-------------+----------------------+--------------------------+----------------------+----------------+---------------+--------------+--------------------+-------------+-------------------+
|id_transacao|id_cliente|id_produto|     data_transacao|quantidade|preco_unitario|valor_total|desconto_aplicado|     status|metodo_pagamento|custo_envio|data_cadastro|idade|   genero|          pais|           segmento|cancelou|valor_vitalicio|possui_app|  categoria|     marca|preco_catalogo_produto|     dispositivo|           canal|total_sessoes|media_duracao_segundos|total_paginas_visualizadas|total_adicoes_carrinho|total_conversoes|total_rejeicoes|avaliacao_nota|avalia

In [ ]:
df_tratado.write.mode('overwrite').option('header',True).parquet('drive/MyDrive/trabalho_ebac_pyspark/02_dados_tratados/vendass_consolidadas.parquet')
print("🚀 Tabela profissional salva com sucesso na pasta 02_dados_tratados!")

🚀 Tabela profissional salva com sucesso na pasta 02_dados_tratados!


In [ ]:
df_tratado.select(
    F.min("data_transacao").alias("primeira_data"),
    F.max("data_transacao").alias("ultima_data")
).show()

+-------------------+-------------------+
|      primeira_data|        ultima_data|
+-------------------+-------------------+
|2023-01-01 00:23:39|2024-12-30 23:59:05|
+-------------------+-------------------+

